## Notebook 2: Inferring Stellar Temperatures from Broadband Photometry

**Audience:** First-year PhD students in Astrophysics  
**Theme:** Statistical inference from noisy astronomical observations

###### Astrophysical context

Large photometric surveys (e.g. SDSS, Gaia, Pan-STARRS) provide flux measurements across multiple wavelength bands for millions of stars.

From these observations, we aim to infer physical stellar properties such as the effective temperature $T_{\rm eff}$.

Spectroscopy provides direct access to these parameters but is observationally expensive, limiting its use for large-scale surveys.

Photometry is far cheaper but indirect, leading to an inverse problem:

> infer stellar physical parameters from broadband colours alone.

###### Physical basis

Stellar colours encode information about the spectral energy distribution.

Hotter stars emit more flux at short wavelengths, while cooler stars peak at longer wavelengths. This produces a monotonic but nonlinear mapping between colour indices and temperature.

We therefore use:

$(u-g), (g-r), (r-i), (i-z)$

as features encoding this spectral shape.

###### Objective

We treat stellar parameter estimation as a statistical regression problem:

- input: photometric colours
- output: $\log_{10}(T_{\rm eff})$

with the goal of learning a mapping that is:
- nonlinear
- robust to noise
- statistically well-validated

rather than relying on simple empirical colour–temperature relations.

###### Core scientific idea

Most inference problems in observational astrophysics can be viewed as:

> physical signal + observational noise + statistical modelling

The challenge is not simply fitting a model, but extracting reliable physical information from incomplete and noisy measurements while preserving scientific interpretability.

## Imports

We use standard scientific Python tools:

- NumPy / Pandas: numerical computation and data handling  
- Matplotlib / Seaborn: visualization  
- scikit-learn: regression models, preprocessing, and evaluation  

These provide a complete pipeline for data analysis and statistical modelling.

In [ ]:
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import os
from sklearn.model_selection import train_test_split,cross_validate, KFold, RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## Reproducibility

We fix the random seed to ensure consistent results across runs.

This guarantees reproducible:
- train/test splits
- model initialization
- synthetic or stochastic behaviour

Reproducibility is essential for scientific validation and comparison of models.

In [ ]:
SEED = 42
np.random.seed(SEED)

# Optional advanced sklearn configuration (only needed if metadata routing is used later)
sklearn.set_config(enable_metadata_routing=True)

# Plot styling chosen for readability and colorblind accessibility
plt.style.use("tableau-colorblind10")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 12

## Data loading

We load the dataset and perform basic validation to ensure:

- required columns are present  
- missing values are removed  
- only physically valid samples are retained  

This ensures that subsequent analysis is not affected by malformed or incomplete data.

In [ ]:
def load_and_validate_stellar_data(file_path):
    """
    Loads stellar photometry and performs multi-layer validation.
    """
    # 1. Systemic Error Handling (File Presence)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Critical Error: The file '{file_path}' was not found.")

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        raise IOError(f"Failed to parse CSV: {e}")

    # 2. Structural Validation (Column presence)
    # These are essential for color-temperature inference
    required_columns = ["Teff", "u_g", "g_r", "r_i", "i_z", "feh", "sigma_phot"]
    missing_cols = [col for col in required_columns if col not in df.columns]
    
    if missing_cols:
        raise KeyError(f"Dataset is missing required astronomical features: {missing_cols}")

    # 3. Physical Validation (Filtering Non-Physical Artifacts)
    initial_count = len(df)

    # Drop any rows that have NaN in key features
    df = df.dropna(subset=required_columns)

    final_count = len(df)
    
    if final_count == 0:
        raise ValueError("Data validation failed: No valid stellar samples remain after filtering.")
    
    print(f"Successfully loaded {final_count} stars ({initial_count - final_count} rows discarded).")
    return df

    

In [ ]:
# Execution
try:
    df = load_and_validate_stellar_data("../data/sdss_processed_colors_v1.csv")
except Exception as err:
    print(f"Pipeline Halted: {err}")


## Target transformation

In astronomy, stellar effective temperature is commonly expressed on a logarithmic scale. 

Metal-rich stellar atmospheres contain more absorption lines, especially in the blue and ultraviolet.

This changes the broadband spectral energy distribution and therefore shifts stellar colours even at fixed effective temperature.


In [ ]:
# Temperature often behaves more smoothly in logarithmic space.
# - relative errors matter more than absolute Kelvin errors
# - distributions are often better behaved
# - multiplicative trends become additive

df["log_Teff"] = np.log10(df["Teff"])


## Train–test split

We split the dataset into training and test sets:

- training set: used to fit model parameters  
- test set: held out for final evaluation  

The test set is never used during model selection to ensure unbiased performance estimates.

The target is defined as:

$y = \log_{10}(T_{\rm eff})$

We use four colour indices as features:

$(u-g), (g-r), (r-i), (i-z)$

These provide a compact representation of the stellar spectral energy distribution.

In [ ]:
# Four colours help break degeneracies caused by metallicity and residual observational systematics.

features = ["u_g", "g_r", "r_i", "i_z"]
X = df[features]
y = df["log_Teff"]


In [ ]:
# test_size=0.25: You are hiding 25% of your stars in the test set
# random_state=SEED: This makes the experiment reproducible and we get the exact same stars in their training set.

X_train, X_test, y_train, y_test, sig_train, sig_test = train_test_split(
    X,
    y,
    df["sigma_phot"],
    test_size=0.25,
    random_state=SEED
)

print("Training stars:", len(X_train))
print("Test stars:", len(X_test))


## Inverse-variance weighting

Photometric measurements have heterogeneous uncertainties.

Inverse-variance weighting is commonly used in astronomy to reduce the influence of noisy observations.

Strictly speaking, weighted least squares is statistically optimal when the target uncertainties are known. Here we instead use photometric uncertainties as a practical proxy for overall observational reliability.

Measurements with smaller variance carry more statistical information and therefore contribute more strongly to the likelihood.

We account for this by weighting samples as:

$w = \frac{1}{\sigma_{\text{phot}}^2}$

This ensures that high-quality measurements contribute more to the loss function, while noisy observations are down-weighted.

Weights are normalized to maintain numerical stability.


In [ ]:
def weighted_rmse(y_true, y_pred, sample_weight=None):
    """
    Weighted root mean squared error.
    """

    if sample_weight is None:
        return np.sqrt(
            mean_squared_error(y_true, y_pred)
        )

    return np.sqrt(
        np.average(
            (y_true - y_pred) ** 2,
            weights=sample_weight
        )
    )


In [ ]:
weighted_rmse_scorer = make_scorer(
    weighted_rmse,
    greater_is_better=False
)

# IMPORTANT:
# Explicitly request sample_weight routing
weighted_rmse_scorer.set_score_request(
    sample_weight=True
)

In [ ]:
# Avoid division by zero
sig_train_safe = np.clip(sig_train, 1e-4, None)
w_train = 1.0 / sig_train_safe**2
# Normalise weights (important for some models)
w_train = w_train / np.mean(w_train)
sig_test_safe = np.clip(sig_test, 1e-4, None)

w_test = 1.0 / sig_test_safe**2
w_test = w_test / np.mean(w_test)

## Utility functions

High $R^2$ does not necessarily imply physically unbiased predictions. 

Residual structure and systematic trends may still remain.

In [ ]:
def evaluate_model(
    name,
    y_true,
    y_pred,
    sample_weight=None
):

    rmse = weighted_rmse(
        y_true,
        y_pred,
        sample_weight
    )

    mae = mean_absolute_error(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    r2 = r2_score(
        y_true,
        y_pred,
        sample_weight=sample_weight
    )

    return {
        "Model": name,
        "RMSE [dex]": rmse,
        "MAE [dex]": mae,
        "R2": r2
    }

## Linear regression baseline

We first fit a linear model as a baseline estimator:

- assumes a linear mapping between colours and temperature  
- provides a reference for more complex models  

While physically interpretable, this model cannot capture nonlinear colour–temperature relations and serves primarily as a benchmark.

In [ ]:
lin4_model = LinearRegression()
lin4_model.fit(X_train, y_train, sample_weight=w_train)

lin4_pred = lin4_model.predict(X_test)

print(pd.Series(
    evaluate_model("Linear (4 colours)", y_test, lin4_pred,sample_weight=w_test)
))


## Decision trees

Decision trees partition feature space into regions and assign constant predictions within each region.

They can model:
- nonlinear dependencies
- population structure (e.g. dwarfs vs giants)
- interactions between colour indices

However, they are prone to overfitting, so regularisation is applied via:
- maximum depth
- minimum samples per leaf

Trees provide a local, rule-based approximation of the colour–temperature relation.


Inverse-variance weighting arises naturally from maximum-likelihood estimation under Gaussian observational uncertainties. 

Measurements with smaller variance carry more statistical information and therefore contribute more strongly to the likelihood.


In [ ]:
tree1_model = DecisionTreeRegressor(max_depth=2)
tree1_model.fit(X_train[["g_r"]], y_train, sample_weight=w_train)

tree1_pred = tree1_model.predict(X_test[["g_r"]])


In [ ]:
sklearn.tree.plot_tree(tree1_model)

In [ ]:
plt.figure()
plt.scatter(X_test["g_r"], y_test, s=1, edgecolor="black", c="darkorange", label="data")
plt.scatter(X_test["g_r"], tree1_pred, s=1, color="yellowgreen", label="max_depth=2")
plt.xlabel(r"$g-r$")
plt.ylabel(r"$\log T_{eff}$")
plt.title("Decision Tree Regression")
plt.legend()
plt.show()

##  Decision Trees Regressor using all colors

In [ ]:
tree4_model = DecisionTreeRegressor(
    max_depth=6,
    min_samples_leaf=20,
    min_samples_split=40,
    random_state=42
)

tree4_model.fit(X_train, y_train, sample_weight=w_train)

tree4_pred = tree4_model.predict(X_test)

print(pd.Series(
    evaluate_model("Decision Tree (all colours)", y_test, tree4_pred,
    sample_weight=w_test)
))


## Random forest

Random forests combine many decision trees trained on bootstrap samples.

Each tree is trained on a slightly different dataset, and predictions are averaged.

Specifically, Random Forest use:

*   **Bootstrap Sampling:** Each tree is trained on a random "reshuffle" of your stars. <br>If a few stars have bad data, they only affect a small fraction of the forest, keeping the overall model stable.
*   **Random Feature Subsets:** At every split, the tree is forced to choose from a random "grab bag" of colors ($u-g$, $r-i$, etc.).<br> This prevents the model from relying *only* on the easiest feature ($g-r$) and forces it to learn the subtle physics hidden in the other bands.

This reduces variance and improves generalisation while retaining nonlinear flexibility.

The result is a more stable estimator of the colour–temperature relation compared to a single decision tree.


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=10,
    min_samples_leaf=10,
    min_samples_split=20,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train, sample_weight=w_train)

rf_pred = rf_model.predict(X_test)

print(pd.Series(
    evaluate_model("Random Forest (all colours)", y_test, rf_pred,
    sample_weight=w_test)
))

## Extreme Gradient Boosting (XGBoost)

Extreme Gradient Boosting builds trees sequentially, where each new tree corrects the residual errors of the previous ensemble.

This allows the model to:
- reduce bias incrementally  
- capture weak nonlinear structure  
- improve predictive accuracy on structured tabular data  

Compared to random forests, boosting typically achieves higher accuracy but requires more careful regularisation.

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.0,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train,
    sample_weight=w_train
)

xgb_pred = xgb_model.predict(X_test)

print(pd.Series(
    evaluate_model("XGBoost", y_test, xgb_pred,
    sample_weight=w_test)
))



In [ ]:
results = pd.DataFrame([
    evaluate_model("Linear (4 colours)", y_test, lin4_pred,
    sample_weight=w_test),
    evaluate_model("Decision Tree", y_test, tree4_pred,
    sample_weight=w_test),
    evaluate_model("Random Forest", y_test, rf_pred,
    sample_weight=w_test),
    evaluate_model("XGBoost", y_test, xgb_pred,
    sample_weight=w_test)
])

results = results.sort_values("RMSE [dex]")
results


## Model selection and cross-validation

Machine learning models involve two types of quantities:

- **Parameters:** learned from data during training (e.g. weights, tree splits)  
- **Hyperparameters:** chosen before training and control model complexity (e.g. depth, learning rate)  

> Parameters are learned; hyperparameters are chosen.

Using the test set for hyperparameter tuning leads to **data leakage**, producing overly optimistic performance estimates that do not generalise.

To avoid this, we separate the workflow into:

- **Training set:** used to learn model parameters  
- **Validation set:** used to tune hyperparameters and select models  
- **Test set:** used once at the end for an unbiased final evaluation  

The test set must remain completely unseen until the final evaluation step.

###### K-fold cross-validation

A single train/test split can produce noisy or unstable estimates depending on how the data is partitioned.

K-fold cross-validation provides a more robust evaluation procedure:

- Split the dataset into \(K\) subsets (folds)  
- Train on \(K-1\) folds and validate on the remaining fold  
- Repeat this process \(K\) times, each fold serving once as validation  
- Average the performance across all folds  

This reduces sensitivity to any particular data split and yields a more reliable estimate of generalisation performance.

###### Implementation detail (scikit-learn)

Scikit-learn maximises scores by default, so metrics such as RMSE are negated (e.g. `neg_root_mean_squared_error`).

- Lower RMSE is better → converted to a negative score for compatibility  
- Final results are converted back to RMSE for interpretation  

###### Interpretation

- Low variance across folds → stable, well-generalising model  
- High variance across folds → model may be unstable or overfitting  

Cross-validation therefore provides a more reliable estimate of predictive performance than a single train/test split by averaging results over multiple partitions of the data.

In [ ]:
cv_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler().set_fit_request(sample_weight=False)),
    ("reg", Ridge(alpha=1.0).set_fit_request(sample_weight=True))
])

#cv_results = cross_validate(
#    cv_model, X_train, y_train, cv=5, scoring = weighted_rmse_scorer,
#    scoring="neg_root_mean_squared_error",
#    params={'sample_weight': w_train},
#    return_train_score=True  
#)

cv_results = cross_validate(
    cv_model,
    X_train,
    y_train,
    cv=5,
    scoring=weighted_rmse_scorer,
    params={"sample_weight": w_train},
    return_train_score=True
)

print("Train RMSE:", -cv_results['train_score'])
print("Val   RMSE:", -cv_results['test_score'])
# I commenter out since weights are computed on the full training set into CV folds.
# The leakage impact is tiny because weights depend only on local uncertainties, not targets,
# but conceptually this is still fold-global information.

In [ ]:
cv_model_retrain = sklearn.base.clone(cv_model)

cv_model_retrain.fit(
    X_train,
    y_train,
    sample_weight=w_train 
)

cv_pred = cv_model_retrain.predict(X_test)

res = evaluate_model("Ridge Poly", y_test, cv_pred,sample_weight=w_test)

print("Test RMSE:", round(res["RMSE [dex]"], 4))


## Random hyperparameter search across all models

To compare multiple candidate regressors fairly, we perform a **randomized hyperparameter search** over both model types and their associated regularisation/complexity parameters.

This approach is preferred over manual tuning because it:

- efficiently explores a wide hyperparameter space  
- applies consistent cross-validation across all models  
- reduces implicit researcher bias in model selection  
- often identifies strong configurations with fewer evaluations than exhaustive grid search  

In practice, this procedure treats model choice and hyperparameter tuning as a single unified optimisation problem.

For stellar calibration tasks with heteroscedastic noise, it is important that the scoring function reflects the scientific objective. Common choices include weighted RMSE, MAE, or \(R^2\), depending on whether the goal is accuracy, robustness, or explained variance.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = weighted_rmse_scorer

In [ ]:
def poly_pipeline(reg):
    return Pipeline([
        ("poly", PolynomialFeatures(include_bias=False)),
        ("scale", StandardScaler().set_fit_request(sample_weight=False)),
        ("reg", reg.set_fit_request(sample_weight=True))
    ])


models = {
    # -------------------------
    # Linear family (poly)
    # -------------------------
    "poly_linear": (
        poly_pipeline(LinearRegression()),
        {
            "poly__degree": [1,2,3,4]
        }
    ),

    "poly_ridge": (
        poly_pipeline(Ridge()),
        {
            "poly__degree": stats.randint(1, 5),
            "reg__alpha": stats.loguniform(1, 1e4)
        }
    ),

    "poly_lasso": (
        poly_pipeline(Lasso(max_iter=1000000)),
        {
            "poly__degree": stats.randint(1, 5),
            "reg__alpha": stats.loguniform(1e-5, 1e-1)
        }
    ),

    # -------------------------
    # Trees
    # -------------------------
    "tree": (
        DecisionTreeRegressor(random_state=42).set_fit_request(sample_weight=True),
        {
            "max_depth": stats.randint(2, 20),
            "min_samples_leaf": stats.randint(2, 50),
            "min_samples_split": stats.randint(2, 100)
        }
    ),

    # -------------------------
    # Random Forest
    # -------------------------
    "rf": (
        RandomForestRegressor(
            random_state=42,
            n_jobs=1
        ).set_fit_request(sample_weight=True),
        {
            "n_estimators": stats.randint(300, 800),
            "max_depth": stats.randint(3, 25),
            "min_samples_leaf": stats.randint(1, 30),
            "min_samples_split": stats.randint(2, 80),
            "max_features": ["sqrt", "log2", None]
        }
    ),

    # -------------------------
    # XGBoosting
    # -------------------------
    "xgb": (
    XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=1
    ).set_fit_request(sample_weight=True),
    {
        "n_estimators": stats.randint(300, 1200),
        "learning_rate": stats.loguniform(0.005, 0.1),
        "max_depth": stats.randint(3, 8),
        "subsample": stats.uniform(0.6, 0.4),
        "colsample_bytree": stats.uniform(0.6, 0.4),
        "min_child_weight": stats.randint(1, 15),
        "reg_alpha": stats.loguniform(1e-5, 10),
        "reg_lambda": stats.loguniform(1e-3, 100)
    }
    )
}

In [ ]:
results = []

for name, (model, params) in models.items():
    if name=="poly_linear":
        search = GridSearchCV(
            estimator=model,
            param_grid=params,
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            verbose=1

        )
    else:    
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=params,
            n_iter=200,
            scoring=scoring,
            cv=cv,
            n_jobs=-1,
            random_state=42,
            verbose=1
        )

    search.fit(X_train, y_train, sample_weight=w_train)

    best_model = search.best_estimator_
    pred = best_model.predict(X_test)

    metrics = evaluate_model(name, y_test, pred, sample_weight=w_test)

    results.append({
        **metrics,
        "best_cv_score": -search.best_score_,
        "best_params": search.best_params_
    })

results_df = pd.DataFrame(results)


In [ ]:
print(results_df[["Model", "RMSE [dex]", "MAE [dex]", "R2", "best_cv_score"]].sort_values("best_cv_score"))


In [ ]:
for model_name, params in zip(results_df["Model"], results_df["best_params"]):
    print(f"{model_name}:")
    for k,v in params.items():
        print(f"\t{k}: {v}")
    print()

### Parity Plot (One-to-One)

This plot compares predicted values against true values to assess how well the model reproduces the underlying physical relationship.

In an ideal case, all points lie exactly on the 45° diagonal line, indicating perfect predictions.

###### 45° line (ideal reference)

- Points on the line → correct prediction  
- Points above the line → underprediction  
- Points below the line → overprediction  

###### Bias

Systematic deviations from the diagonal indicate model bias.

- Curvature away from the line, especially at the extremes, suggests the model performs poorly on rare or extreme values  
- Consistent offset from the diagonal indicates a systematic error in calibration  

###### Spread

The dispersion of points around the diagonal reflects predictive accuracy:

- tight clustering → low error, high precision  
- wide scatter → higher uncertainty and poorer performance  

###### Why this plot is useful

Unlike single summary metrics (e.g. RMSE or $R^2$), the parity plot reveals:

- where the model performs well or poorly across the target range  
- whether errors are systematic or random  
- how performance changes at the extremes of the distribution  

In [ ]:
rf_best_result = next(r for r in results if r["Model"] == "rf")
best_rf_params = rf_best_result["best_params"]

rf_best = RandomForestRegressor(
    **best_rf_params,
    random_state=42,
    n_jobs=-1
)

rf_best.fit(X_train, y_train, sample_weight=w_train)

rf_best_pred = rf_best.predict(X_test)


In [ ]:
plt.scatter(rf_best_pred, y_test, s=10, alpha=0.3)
lims = [min(y_test.min(), rf_best_pred.min()),
        max(y_test.max(), rf_best_pred.max())]

plt.plot(lims, lims, '--', color='black')
plt.xlabel(r"Predicted $\log(T_{eff})$")
plt.ylabel(r"True $\log(T_{eff})$")
plt.title("Random Forest Mean Prediction")
plt.tight_layout()
plt.show()


## Residual analysis

Residuals are defined as:

$residual = y - \hat{y}$

They reveal structure not captured by global metrics such as RMSE or $R^2$.

A good model should produce residuals that are:
- centered around zero  
- uncorrelated with input features  
- approximately structureless  

Deviations indicate missing structure or model misspecification.

In [ ]:
#resid = pred_ridge - y_test
resid = y_test - rf_best_pred
fig, ax = plt.subplots(1, 2, figsize=(12,4))

ax[0].scatter(y_test, resid, s=10, alpha=0.3)
ax[0].axhline(0, ls="--")
ax[0].set_xlabel("True log(Teff)")
ax[0].set_ylabel("Residual [dex]")
ax[0].set_title("Residual Trend")

ax[1].hist(resid, bins=30)
ax[1].set_xlabel("Residual [dex]")
ax[1].set_title("Residual Distribution")

plt.tight_layout()
plt.show()



## Residual dependence on metallicity

We test whether residuals depend on stellar metallicity $[{\rm Fe/H}]$.

If residuals show structure with respect to metallicity, this indicates that:
- metallicity affects stellar colours  
- the model cannot fully disentangle temperature and composition  

This reveals the presence of a latent physical variable not included in the feature set, highlighting a fundamental limitation of purely photometric inference.

In [ ]:
feh_test = df.loc[X_test.index, "feh"]

plt.scatter(feh_test, resid, s=10, alpha=0.3)
plt.axhline(0, ls="--")
plt.xlabel("[Fe/H]")
plt.ylabel("Residual [dex]")
plt.title("Residuals vs Metallicity")
plt.tight_layout()
plt.show()


## Scientific interpretation

This notebook explored how stellar effective temperatures can be inferred from broadband photometric measurements using a sequence of increasingly flexible regression models.

Rather than treating machine learning as a purely computational exercise, the notebook framed stellar parameter estimation as a scientific inference problem involving observational uncertainty, hidden variables, and model validation.

### Main scientific conclusions

###### Photometric colours contain real physical information

Broadband colours are directly connected to the stellar spectral energy distribution and therefore encode information about temperature.

Even relatively simple colour indices provide meaningful constraints on: $T_{\rm eff}$

because hotter stars systematically appear bluer while cooler stars appear redder.

The success of the regression models demonstrates that this physical relationship is measurable even in the presence of observational noise.

###### Multi-band photometry improves parameter inference

Single-colour estimators suffer from degeneracies:
different physical effects can produce similar colours.

Using multiple filters simultaneously improves inference by helping disentangle competing effects such as:

- stellar temperature
- metallicity effects
- residual calibration

This reflects a broader principle in astronomy:

> richer observational information reduces physical degeneracy.

###### Nonlinear models better capture astrophysical structure

The relationship between stellar colours and temperature is not perfectly linear.

Tree-based models and ensemble methods outperform simple linear regression because they can represent:

- curved colour–temperature relations
- multimodal stellar populations
- weak nonlinear interactions between observables

Flexible models therefore recover astrophysical structure that linear approximations cannot fully capture.

###### Noise modelling matters

Astronomical measurements are heteroscedastic:
different stars are observed with different uncertainties.

Inverse-variance weighting improves statistical efficiency by giving greater importance to higher-quality measurements while down-weighting noisy observations.

This is a core principle of observational astrophysics and statistical inference more generally.

###### Validation is essential for trustworthy science

A model that performs well on its training data may still fail catastrophically on unseen observations.

Cross-validation and held-out test sets are therefore necessary to distinguish:

- genuine physical learning
from
- memorisation of statistical noise

Reliable astrophysical machine learning requires rigorous separation between training, tuning, and evaluation.

###### Residuals reveal hidden physics

Residual diagnostics showed that prediction errors are not entirely random.

Correlations between residuals and metallicity demonstrate that stellar chemistry affects photometric colours in ways not fully captured by colour information alone.

This highlights an important scientific lesson:

> systematic residual structure often indicates missing physics.

Residual analysis is therefore not merely a debugging tool;
it is a method for discovering limitations in the physical model itself.

### Broader interpretation

This notebook illustrates a broader paradigm used throughout modern astrophysics:

- infer latent physical quantities
- from noisy indirect observables
- using probabilistic statistical models
- validated through rigorous testing

The methodology extends far beyond stellar temperatures and applies to problems such as:

- photometric redshift estimation
- galaxy classification
- exoplanet characterisation
- cosmological parameter inference
- transient event detection

### Final takeaway

Stellar parameter estimation is fundamentally an inference problem under uncertainty.

Successful astrophysical machine learning requires combining:

- physical understanding
- statistical rigour
- uncertainty quantification
- and careful validation

The objective is not simply to obtain accurate predictions, but to build models whose behaviour remains scientifically interpretable, statistically reliable, and physically meaningful when applied to real observational data.